# 🧪 Lab — Can You Trust This Dataset?

## Overview

You have been given a dataset containing e-commerce sales data.

Your manager has already used this dataset to report revenue metrics to leadership.

Your job is to determine:

> **Can this dataset be trusted for analysis?**

---

## IMPORTANT

To get the dataset, navigate to the `./data/raw/` folder. 

You should see a Python file called: `generate_sales_data.py`.

**IN YOUR TERMINAL** inside the `./data/raw/` folder run the following command:

```
python3 generate_sales_data.py
```

You should then have a `sales.csv` file in the folder after it runs.

In [3]:
# Run your imports here`
import pandas as pd
import numpy as np
from pathlib import Path

Load the dataset.

Name the dataframe `sales`

In [5]:
"""
generate_sales_data.py

Generates a messy e-commerce sales dataset for a data validation lab.
"""

import pandas as pd
import numpy as np


def generate_sales_data(n=300, seed=42):
    np.random.seed(seed)

    sales = pd.DataFrame({
        "order_id": np.arange(1000, 1000 + n),
        "customer_id": np.random.randint(1, 80, size=n),
        "product_id": np.random.randint(200, 230, size=n),
        "revenue": np.round(np.random.normal(loc=200, scale=75, size=n), 2),
        "quantity": np.random.randint(1, 6, size=n),
        "unit_price": np.round(np.random.normal(loc=50, scale=15, size=n), 2),
        "status": np.random.choice(
            ["Complete", "Cancelled", "Returned"],
            size=n,
            p=[0.72, 0.18, 0.10]
        ),
        "payment_method": np.random.choice(
            ["Credit Card", "Debit Card", "PayPal", "Gift Card"],
            size=n
        ),
        "region": np.random.choice(
            ["West", "East", "South", "Midwest"],
            size=n
        )
    })

    sales["order_date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(
        np.random.randint(0, 365, size=n), unit="D"
    )

    # --------------------------------------------------
    # Obvious data-quality issues
    # --------------------------------------------------

    duplicate_indices = np.random.choice(sales.index, size=12, replace=False)
    sales.loc[duplicate_indices, "order_id"] = sales.loc[duplicate_indices, "order_id"] - 1

    neg_revenue_indices = np.random.choice(sales.index, size=8, replace=False)
    sales.loc[neg_revenue_indices, "revenue"] = -abs(sales.loc[neg_revenue_indices, "revenue"])

    zero_quantity_indices = np.random.choice(sales.index, size=6, replace=False)
    sales.loc[zero_quantity_indices, "quantity"] = 0

    missing_customer_indices = np.random.choice(sales.index, size=12, replace=False)
    sales.loc[missing_customer_indices, "customer_id"] = np.nan

    future_date_indices = np.random.choice(sales.index, size=6, replace=False)
    sales.loc[future_date_indices, "order_date"] = pd.to_datetime("2035-01-01")

    outlier_indices = np.random.choice(sales.index, size=5, replace=False)
    sales.loc[outlier_indices, "revenue"] *= 25

    # --------------------------------------------------
    # Messy categorical issues
    # --------------------------------------------------

    status_noise = [
        "complete",
        " COMPLETE ",
        "completed",
        "Cancelled ",
        "CANCELLED",
        "return",
        "Returned ",
        "unknown"
    ]

    status_indices = np.random.choice(sales.index, size=25, replace=False)
    sales.loc[status_indices, "status"] = np.random.choice(status_noise, size=25)

    payment_noise = [
        "credit card",
        "CreditCard",
        " CREDIT CARD ",
        "paypal",
        "Pay Pal",
        "giftcard",
        "Unknown"
    ]

    payment_indices = np.random.choice(sales.index, size=20, replace=False)
    sales.loc[payment_indices, "payment_method"] = np.random.choice(payment_noise, size=20)

    region_noise = [
        "west",
        "WEST ",
        "east",
        " south ",
        "Mid-West",
        "Northeast",
        None
    ]

    region_indices = np.random.choice(sales.index, size=18, replace=False)
    sales.loc[region_indices, "region"] = np.random.choice(region_noise, size=18)

    # --------------------------------------------------
    # Hidden edge cases
    # --------------------------------------------------

    # 1. Revenue does not match quantity * unit_price
    mismatch_indices = np.random.choice(sales.index, size=20, replace=False)
    sales.loc[mismatch_indices, "revenue"] = np.round(
        sales.loc[mismatch_indices, "revenue"] * np.random.uniform(0.5, 1.8, size=20),
        2
    )

    # 2. Unit price is negative or zero
    bad_price_indices = np.random.choice(sales.index, size=6, replace=False)
    sales.loc[bad_price_indices[:3], "unit_price"] = 0
    sales.loc[bad_price_indices[3:], "unit_price"] = -abs(sales.loc[bad_price_indices[3:], "unit_price"])

    # 3. Cancelled orders still have large positive revenue
    cancelled_revenue_indices = sales[sales["status"] == "Cancelled"].sample(
        n=min(8, (sales["status"] == "Cancelled").sum()),
        random_state=seed
    ).index

    sales.loc[cancelled_revenue_indices, "revenue"] = np.round(
        np.random.uniform(150, 600, size=len(cancelled_revenue_indices)),
        2
    )

    # 4. Returned orders have quantity but no negative/adjusted revenue
    returned_indices = sales[sales["status"] == "Returned"].sample(
        n=min(6, (sales["status"] == "Returned").sum()),
        random_state=seed + 1
    ).index

    sales.loc[returned_indices, "revenue"] = np.round(
        np.random.uniform(100, 500, size=len(returned_indices)),
        2
    )

    # 5. Duplicate full rows, not just duplicate IDs
    full_duplicate_rows = sales.sample(n=5, random_state=seed + 2)
    sales = pd.concat([sales, full_duplicate_rows], ignore_index=True)

    # 6. Whitespace in column names
    sales = sales.rename(columns={
        "payment_method": "payment_method ",
        "region": " region"
    })

    # 7. A numeric-looking column stored as strings
    string_revenue_indices = np.random.choice(sales.index, size=10, replace=False)
    sales.loc[string_revenue_indices, "revenue"] = sales.loc[string_revenue_indices, "revenue"].astype(str)

    # 8. A few revenue values with dollar signs
    dollar_indices = np.random.choice(sales.index, size=5, replace=False)
    sales.loc[dollar_indices, "revenue"] = "$" + sales.loc[dollar_indices, "revenue"].astype(str)

    # 9. A few impossible customer IDs
    bad_customer_indices = np.random.choice(sales.index, size=4, replace=False)
    sales.loc[bad_customer_indices, "customer_id"] = -1

    # 10. A few invalid product IDs
    bad_product_indices = np.random.choice(sales.index, size=4, replace=False)
    sales.loc[bad_product_indices, "product_id"] = 9999

    return sales


if __name__ == "__main__":
    df = generate_sales_data()
    df.to_csv("sales.csv", index=False)
    print("sales.csv generated successfully.")

sales.csv generated successfully.


C:\Users\Greg C\AppData\Local\Temp\ipykernel_27336\3540171677.py:156: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['97.73' '218.37' '-131.93' '262.68' '183.89' '399.22' '132.71' '347.35'
 '160.21' '175.55']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  sales.loc[string_revenue_indices, "revenue"] = sales.loc[string_revenue_indices, "revenue"].astype(str)


In [6]:
sales = pd.read_csv('sales.csv')

In [7]:
sales.columns = sales.columns.str.strip()

In [8]:
sales.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07
2,1002,72.0,209,286.86,1,65.15,Returned,Credit Card,West,2023-11-26
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08
4,1004,21.0,225,246.81,1,33.59,Returned,Gift Card,West,2023-05-24


In [9]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        305 non-null    int64  
 1   customer_id     293 non-null    float64
 2   product_id      305 non-null    int64  
 3   revenue         305 non-null    object 
 4   quantity        305 non-null    int64  
 5   unit_price      305 non-null    float64
 6   status          305 non-null    object 
 7   payment_method  305 non-null    object 
 8   region          305 non-null    object 
 9   order_date      305 non-null    object 
dtypes: float64(2), int64(3), object(5)
memory usage: 24.0+ KB


In [10]:
sales.describe()


,order_id,customer_id,product_id,quantity,unit_price
count,305.000000,293.000000,305.000000,305.000000,305.000000
mean,1149.363934,38.709898,343.629508,2.819672,49.615607
std,86.692792,23.068268,1114.916428,1.458958,20.288912
min,1000.000000,-1.000000,200.000000,0.000000,-96.570000
25%,1075.000000,20.000000,208.000000,2.000000,40.240000
50%,1149.000000,38.000000,216.000000,3.000000,49.600000
75%,1224.000000,60.000000,223.000000,4.000000,60.170000
max,1299.000000,79.000000,9999.000000,5.000000,97.510000


In [11]:
sales[sales["customer_id"].isna()]

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date
36,1036,NaN,225,238.95,3,57.03,Complete,Credit Card,east,2023-03-07
55,1055,NaN,221,139.38,4,53.44,Complete,Credit Card,Midwest,2023-06-13
59,1059,NaN,229,$160.57,5,40.54,Cancelled,Debit Card,East,2023-02-16
60,1060,NaN,228,143.07,3,63.61,Cancelled,Gift Card,East,2023-08-26
85,1085,NaN,219,238.58,4,36.77,Complete,Credit Card,Mid-West,2023-05-17
92,1092,NaN,215,316.34,3,58.36,Cancelled,Gift Card,South,2023-01-21
100,1100,NaN,223,248.4,4,41.20,Cancelled,Gift Card,West,2023-09-04
186,1186,NaN,222,220.28,5,46.67,Complete,PayPal,Midwest,2023-04-01
217,1217,NaN,218,115.27,3,56.18,Complete,Debit Card,West,2023-05-04
281,1281,NaN,218,340.76,3,46.80,Complete,Gift Card,Northeast,2023-08-20


In [12]:
sales["order_id"].duplicated().sum()

17

In [13]:
sales["status"].value_counts()

status
Complete      209
Cancelled      48
Returned       23
Returned        6
CANCELLED       4
unknown         3
Cancelled       3
 COMPLETE       3
complete        3
return          2
completed       1
Name: count, dtype: int64

In [14]:
#sales["status_clean"] = sales["status"].str.lower().str.strip()


In [15]:
sales.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07
2,1002,72.0,209,286.86,1,65.15,Returned,Credit Card,West,2023-11-26
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08
4,1004,21.0,225,246.81,1,33.59,Returned,Gift Card,West,2023-05-24


In [16]:
#sales['status_clean'].value_counts()

In [17]:
sales['payment_method'].value_counts()

payment_method
Gift Card        86
Credit Card      73
Debit Card       66
PayPal           58
paypal            5
Unknown           4
giftcard          4
credit card       3
CreditCard        3
Pay Pal           2
 CREDIT CARD      1
Name: count, dtype: int64

In [18]:
sales['order_id'].is_unique

False

In [19]:
sales['revenue'] = pd.to_numeric(sales['revenue'].replace('[\$,]', '', regex=True), errors='coerce')

<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
C:\Users\Greg C\AppData\Local\Temp\ipykernel_27336\775963742.py:1: SyntaxWarning: invalid escape sequence '\$'
  sales['revenue'] = pd.to_numeric(sales['revenue'].replace('[\$,]', '', regex=True), errors='coerce')


In [20]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        305 non-null    int64  
 1   customer_id     293 non-null    float64
 2   product_id      305 non-null    int64  
 3   revenue         305 non-null    float64
 4   quantity        305 non-null    int64  
 5   unit_price      305 non-null    float64
 6   status          305 non-null    object 
 7   payment_method  305 non-null    object 
 8   region          305 non-null    object 
 9   order_date      305 non-null    object 
dtypes: float64(3), int64(3), object(4)
memory usage: 24.0+ KB


In [21]:
sales[sales["revenue"]<=0]

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date
39,1039,39.0,203,-230.13,4,58.57,Complete,Credit Card,West,2023-02-13
79,1079,41.0,211,-203.52,5,47.71,Returned,Credit Card,South,2023-04-17
102,1102,33.0,227,-127.63,3,66.56,Complete,PayPal,East,2023-10-10
184,1184,32.0,223,-203.64,1,41.30,Cancelled,Gift Card,West,2023-09-08
189,1189,75.0,227,-131.93,2,73.51,Complete,Gift Card,East,2023-02-07
206,1206,67.0,200,-247.89,2,44.95,Cancelled,Debit Card,East,2023-03-24
215,1215,11.0,201,-156.73,3,55.21,Complete,CreditCard,East,2023-03-14
299,1299,67.0,204,-300.91,1,57.61,Cancelled,Credit Card,Midwest,2023-12-27


In [22]:
sales[sales["quantity"]<= 0]

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07
130,1130,36.0,206,160.23,0,87.73,Complete,PayPal,south,2023-06-12
157,1157,9.0,225,128.33,0,56.55,Complete,PayPal,West,2023-05-04
257,1257,20.0,216,168.85,0,20.72,Complete,Gift Card,West,2023-05-29
261,1261,17.0,229,218.62,0,31.10,Cancelled,Debit Card,West,2023-12-23
298,1298,33.0,220,481.34,0,55.60,Cancelled,Gift Card,East,2023-12-13


In [23]:
sales.describe()

,order_id,customer_id,product_id,revenue,quantity,unit_price
count,305.000000,293.000000,305.000000,305.000000,305.000000,305.000000
mean,1149.363934,38.709898,343.629508,268.259115,2.819672,49.615607
std,86.692792,23.068268,1114.916428,590.764366,1.458958,20.288912
min,1000.000000,-1.000000,200.000000,-300.910000,0.000000,-96.570000
25%,1075.000000,20.000000,208.000000,140.710000,2.000000,40.240000
50%,1149.000000,38.000000,216.000000,201.840000,3.000000,49.600000
75%,1224.000000,60.000000,223.000000,256.770000,4.000000,60.170000
max,1299.000000,79.000000,9999.000000,6178.250000,5.000000,97.510000


In [79]:
#get the iqr and make boolean masks for upper and lower bounds to check for negative revenue
q1 = sales["revenue"].quantile(0.25)
q3 = sales["revenue"].quantile(0.75)

iqr = q3 - q1

In [81]:
lower_bound = q1 - 1.5*iqr
upper_bound = q3 + 1.5*iqr

In [83]:
revenue_outliers = sales[
    (sales["revenue"] < lower_bound) | 
    (sales["revenue"] > upper_bound )

]
revenue_outliers

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date
5,1005,75.0,202,6178.25,5,14.43,Cancelled,Gift Card,West,2023-05-05
39,1039,39.0,203,-230.13,4,58.57,Complete,Credit Card,West,2023-02-13
79,1079,41.0,211,-203.52,5,47.71,Returned,Credit Card,South,2023-04-17
102,1102,33.0,227,-127.63,3,66.56,Complete,PayPal,East,2023-10-10
104,1104,23.0,222,450.48,4,40.56,Returned,PayPal,West,2023-01-25
154,1154,70.0,211,430.92,2,53.87,Complete,Credit Card,West,2023-02-16
174,1174,56.0,209,3358.25,3,26.99,Complete,Debit Card,East,2023-05-22
184,1184,32.0,223,-203.64,1,41.30,Cancelled,Gift Card,West,2023-09-08
189,1189,75.0,227,-131.93,2,73.51,Complete,Gift Card,East,2023-02-07
190,1190,56.0,226,3918.50,4,45.53,Cancelled,Credit Card,East,2035-01-01


---

### Initial Analysis

In [93]:
# TODO: Calculate total revenue
#we take teh revenue from all the orders 
#total_rev = sales["revenue"].sum()
# TODO: Calculate average order value
#aov = sales["revenue"].mean()
# TODO: Calculate revenue by status
sales.groupby('status_cleaned')['revenue'].sum()

status
 COMPLETE       626.84
CANCELLED       684.97
Cancelled     24783.40
Cancelled       115.29
Complete      46784.30
Returned       5320.38
Returned       1150.32
complete        764.35
completed       217.67
return          663.90
unknown         707.61
Name: revenue, dtype: float64

In [26]:
#print(f"The  total revenue is ${total_rev}")
#print(f"The total revenue of al${aov}")
#rev_by_status

---

### Data Inspection

Call the usual Pandas methods to inspect a dataset.

Well the revenue data is not numbers they are objects not floats or integers  

---

### Idenfity Issues

Feel free to break this up across multiple cells

# TODO: Check for missing values
There are miising customer_ids
# TODO: Check for duplicates
there are 17 instances of duplicated sums 
# TODO: Check for invalid revenue
revenue is saves os an object instead of a float 
# TODO: Check for invalid quantities
looks like invalid invalid unit prices and reveneues are in teh negatives will need to take teh absolute value of it 
# TODO: Check for inconsistent categories
'status''payment_method' contain capital letters and spaces
# TODO: Check for invalid dates
dates are not in avalid date time will have to be converted todate time

In [85]:
sales["status_clean"] = sales["status"].str.lower().str.strip()
sales["status_clean"].value_counts()

status_clean
complete     215
cancelled     55
returned      29
unknown        3
return         2
completed      1
Name: count, dtype: int64

In [89]:
sales.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date,status_clean
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26,complete
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07,cancelled
2,1002,72.0,209,286.86,1,65.15,Returned,Credit Card,West,2023-11-26,returned
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08,complete
4,1004,21.0,225,246.81,1,33.59,Returned,Gift Card,West,2023-05-24,returned


In [103]:
#converts the orderdate to date 
sales['order_date'] = pd.to_datetime(sales['order_date'], errors='coerce')

In [105]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305 entries, 0 to 304
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        305 non-null    int64         
 1   customer_id     293 non-null    float64       
 2   product_id      305 non-null    int64         
 3   revenue         305 non-null    float64       
 4   quantity        305 non-null    int64         
 5   unit_price      305 non-null    float64       
 6   status          305 non-null    object        
 7   payment_method  305 non-null    object        
 8   region          305 non-null    object        
 9   order_date      305 non-null    datetime64[ns]
 10  status_clean    305 non-null    object        
dtypes: datetime64[ns](1), float64(3), int64(3), object(4)
memory usage: 26.3+ KB


---

### Create Validation Columns

In [107]:
valid_statuses = ["complete", "cancelled","returned"]
today = pd.Timestamp('2026-05-05')

In [109]:
# TODO: Create validation flags
#create an order_id validation flag / customer_id validation flag 

sales["duplicate_order_id"] = sales["order_id"].duplicated(keep=False)
sales["missing_customer_id"] =sales["customer_id"].isna()
sales["negative_revenue"] = sales["revenue"] < 0
sales["invalid_quantity"] = sales["quantity"] <= 0
sales["invalid_status"] = ~sales["status_clean"].isin(valid_statuses)
sales["invalid_date"] = sales["order_date"] > today
# Example:
# sales["missing_customer_id"] = 



sales.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date,status_clean,duplicate_order_id,missing_customer_id,negative_revenue,invalid_quantity,invalid_status,invalid_date
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26,complete,False,False,False,False,False,False
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07,cancelled,False,False,False,True,False,False
2,1002,72.0,209,286.86,1,65.15,Returned,Credit Card,West,2023-11-26,returned,False,False,False,False,False,False
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08,complete,False,False,False,False,False,False
4,1004,21.0,225,246.81,1,33.59,Returned,Gift Card,West,2023-05-24,returned,False,False,False,False,False,False


---

### Summarize Issues



---

The issues t hat we needed to fix is to correct the data for example we need to make sure we dont have duplicated orders
missing customer ids negative reveune negative quantities and correct statuses

In [113]:
# TODO: Summarize validation columns
#create a list of the columns we're usiong as a validation column
validation_columns = [


    "duplicate_order_id",
    "missing_customer_id",
    "negative_revenue",
    "invalid_quantity",
    "invalid_status",
    "invalid_date"
    
]

In [115]:
#sales["has_quality_issue"] = sales[validation_columns]
sales[validation_columns].sum()

sales["has_quality_issue"] = sales[validation_columns].any(axis=1)

problem_rows = sales[sales['has_quality_issue']]

problem_rows.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date,status_clean,duplicate_order_id,missing_customer_id,negative_revenue,invalid_quantity,invalid_status,invalid_date,has_quality_issue
1,1001,15.0,224,108.94,0,75.70,Cancelled,Debit Card,west,2023-03-07,cancelled,False,False,False,True,False,False,True
6,1006,75.0,207,199.08,5,43.44,unknown,Credit Card,Midwest,2023-06-16,unknown,False,False,False,False,True,False,True
12,1012,30.0,222,138.09,5,46.01,Cancelled,Credit Card,East,2023-05-18,cancelled,True,False,False,False,False,False,True
13,1012,38.0,201,175.90,1,40.03,Returned,Gift Card,Midwest,2035-01-01,returned,True,False,False,False,False,True,True
36,1036,NaN,225,238.95,3,57.03,Complete,Credit Card,east,2023-03-07,complete,False,True,False,False,False,False,True


In [122]:
#total revenue and avergae order value and revenue by status
raw_revenue_total = sales["revenue"].sum()
raw_avg_order_value = sales["revenue"].mean()


In [130]:
print(f" the value of the sum of teh reveneue is ${raw_revenue_total} and the average order value is ${raw_avg_order_value}")

 the value of the sum of teh reveneue is $81819.03 and the average order value is $268.25911475409833


In [128]:
sales.groupby('status_clean')['revenue'].sum()

status_clean
cancelled    25583.66
complete     48175.49
completed      217.67
return         663.90
returned      6470.70
unknown        707.61
Name: revenue, dtype: float64

---

### Assertions

In [136]:
# TODO: Write assertions

#tHi sassumes all order_id are unique and will raise an error if not 

#assert sales["order_id"].is_unique,"Order ID's must be unique."

In [138]:
#assert sales["customer_id"].notna().all()," Customer ID cannot be missing "

Thsi sfunction goes through the assert and checvks if the statement is true and sends a error message instead
as we can see here we're getting the "Order ID must be unique" becuase teh assert checks and see that there are order_id duplicates
so and so forth with the rest of the asser statements

In [140]:
def validate_sales_data(df):
    assert df["order_id"].is_unique, "Order ID must be unique"
    assert df["customer_id"].notna().all(), "Customer ID cannot be missing"
    assert (df['revenue'] >= 0).all(), "Revenue cannot be negative"
    assert (df["quantity"] > 0).all(), "Quantity must be greater than zero"

    cleaned_status = df["status"].str.lower().str.strip()
    valid_statuses = ["complete", "cancelled", "returned"]

    assert cleaned_status.isin(valid_statuses).all(), "Status contains an invalid status"

    return True

validate_sales_data(sales)

AssertionError: Order ID must be unique

---

### Now clean your dataset

Save it as `clean_sales`

In [38]:
# TODO: Filter out problematic rows

In [142]:
clean_sales = sales[~sales["has_quality_issue"]].copy()

clean_sales

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date,status_clean,duplicate_order_id,missing_customer_id,negative_revenue,invalid_quantity,invalid_status,invalid_date,has_quality_issue
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26,complete,False,False,False,False,False,False,False
2,1002,72.0,209,286.86,1,65.15,Returned,Credit Card,West,2023-11-26,returned,False,False,False,False,False,False,False
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08,complete,False,False,False,False,False,False,False
4,1004,21.0,225,246.81,1,33.59,Returned,Gift Card,West,2023-05-24,returned,False,False,False,False,False,False,False
5,1005,75.0,202,6178.25,5,14.43,Cancelled,Gift Card,West,2023-05-05,cancelled,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,1293,-1.0,219,254.43,3,41.09,Complete,Debit Card,West,2023-03-17,complete,False,False,False,False,False,False,False
294,1294,53.0,218,236.08,2,48.46,Complete,Debit Card,East,2023-03-08,complete,False,False,False,False,False,False,False
295,1295,60.0,225,216.79,4,49.60,Complete,Debit Card,West,2023-12-08,complete,False,False,False,False,False,False,False
296,1296,63.0,214,140.71,1,60.03,Complete,Debit Card,West,2023-07-13,complete,False,False,False,False,False,False,False


In [144]:
clean_sales.head()

,order_id,customer_id,product_id,revenue,quantity,unit_price,status,payment_method,region,order_date,status_clean,duplicate_order_id,missing_customer_id,negative_revenue,invalid_quantity,invalid_status,invalid_date,has_quality_issue
0,1000,52.0,217,124.55,3,52.18,Complete,Unknown,East,2023-02-26,complete,False,False,False,False,False,False,False
2,1002,72.0,209,286.86,1,65.15,Returned,Credit Card,West,2023-11-26,returned,False,False,False,False,False,False,False
3,1003,61.0,221,259.37,3,47.06,Complete,Credit Card,East,2023-04-08,complete,False,False,False,False,False,False,False
4,1004,21.0,225,246.81,1,33.59,Returned,Gift Card,West,2023-05-24,returned,False,False,False,False,False,False,False
5,1005,75.0,202,6178.25,5,14.43,Cancelled,Gift Card,West,2023-05-05,cancelled,False,False,False,False,False,False,False


In [176]:
clean_rev_by_status = clean_sales.groupby('revenue')['status_clean'].sum()
print(clean_rev_by_status)

revenue
14.63       complete
27.36       complete
34.16      cancelled
40.71       complete
44.94      cancelled
             ...    
537.93     cancelled
3358.25     complete
3867.50     complete
5775.25    cancelled
6178.25    cancelled
Name: status_clean, Length: 236, dtype: object


In [174]:
clean_sales["status_clean"].value_counts()

status_clean
complete     173
cancelled     40
returned      23
Name: count, dtype: int64

---

### Re-run the analysis

Reference the lab `README` to remember which metrics to investigate.

In [151]:
#total revenue and avergae order value and revenue by status
raw_revenue_total = sales["revenue"].sum()
raw_avg_order_value = sales["revenue"].mean()

print(f" the value of the sum of the reveneue before cleaning  is ${raw_revenue_total} and the average order value is ${raw_avg_order_value}")



clean_revenue_total = clean_sales["revenue"].sum()
clean_avg_order_value = clean_sales["revenue"].mean()
print(f" the value of the sum of cleaned reveneue is ${clean_revenue_total} and the average order value is ${clean_avg_order_value}")
 

 the value of the sum of the reveneue before cleaning  is $81819.03 and the average order value is $268.25911475409833
 the value of the sum of cleaned reveneue is $66990.17000000001 and the average order value is $283.85665254237296


---

### Data Quality Report

In [181]:
data_quality_report = sales[validation_columns].sum().sort_values(ascending=False)

print("--- DATA QUALITY REPORT ---")
print("Total Rows Analyzed:", len(sales))
print("Total Problematic Rows Removed:", len(sales) - len(clean_sales))
print("Breakdown of Issues Found:")
print(data_quality_report)

--- DATA QUALITY REPORT ---
Total Rows Analyzed: 305
Total Problematic Rows Removed: 69
Breakdown of Issues Found:
duplicate_order_id     33
missing_customer_id    12
negative_revenue        8
invalid_quantity        6
invalid_status          6
invalid_date            6
dtype: int64


In [172]:
clean_sales.describe()

,order_id,customer_id,product_id,revenue,quantity,unit_price,order_date
count,236.000000,236.000000,236.000000,236.000000,236.000000,236.000000,236
mean,1148.169492,38.508475,339.927966,283.856653,2.843220,48.889153,2023-07-02 01:43:43.728813568
min,1000.000000,-1.000000,200.000000,14.630000,1.000000,-96.570000,2023-01-01 00:00:00
25%,1076.750000,20.000000,209.000000,148.787500,2.000000,39.800000,2023-04-06 18:00:00
50%,1147.500000,38.000000,216.500000,205.625000,3.000000,48.690000,2023-07-09 00:00:00
75%,1222.250000,60.000000,224.000000,258.200000,4.000000,61.310000,2023-09-28 06:00:00
max,1297.000000,79.000000,9999.000000,6178.250000,5.000000,97.510000,2023-12-30 00:00:00
std,84.970313,23.337212,1098.383032,619.776401,1.397869,21.814826,NaN


In [170]:
# TODO: Recalculate metrics
clean_sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 236 entries, 0 to 297
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   order_id             236 non-null    int64         
 1   customer_id          236 non-null    float64       
 2   product_id           236 non-null    int64         
 3   revenue              236 non-null    float64       
 4   quantity             236 non-null    int64         
 5   unit_price           236 non-null    float64       
 6   status               236 non-null    object        
 7   payment_method       236 non-null    object        
 8   region               236 non-null    object        
 9   order_date           236 non-null    datetime64[ns]
 10  status_clean         236 non-null    object        
 11  duplicate_order_id   236 non-null    bool          
 12  missing_customer_id  236 non-null    bool          
 13  negative_revenue     236 non-null    boo

In [ ]:
#data_quality_report = sales[validation_columns].sum().to_frame(name='issue_count')

---

## Export cleaned dataset and report to the appropriate folder

In [192]:
PROCESSED = Path('..')/'data'/ 'processed'
output_path = PROCESSED / "clean_sales.csv"
data_report_path =PROCESSED / "data_report.csv"


clean_sales.to_csv(output_path, index=False)

#make sure to send each on their own file path o whichever one that is sent last wioll overwrite teh other 
data_quality_report.to_csv(data_report_path, index=True)
print("Files saved successfully!")

Files saved successfully!


---

### Reflection Questions.

Answer the reflection questions from the README below in markdown cells.

The issue that had teh biggest impact on teh results is that fore awhile i just could get my head around teh material 
like some of the validation steps I have a btter undewrstanding now sometimes i get confused with how many steps there were 
i also feel likle i have to strat to take mandatory coding breakers because sometimes i feel like i need beautiful soup just to read.
The assumption off the bat is that a lot of this data had to be proceessed and I guess iddnt realize how trustworthy I was to data
I think we have been put on easy mode with all the already cleaned data we get and we only had a state of data cleaning and validation.
Only after running about five runs of .info() describe() and others did I learn taht there were some problems with thsi data albeit I was very 
convicncved to just drop the nas and other duplicate and missing data. Now I see why python warns you so much about making edits tsraight to the raw data is it better to worth with copis, boolena masks and other strategies so I am not directly editing the data amake sure I take the tiem to read teh code and give my brain abreak sometimes.